In [ ]:
import os
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm

def get_all_image_paths(root_dir):
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if os.path.splitext(fname)[1].lower() in valid_ext:
                image_paths.append(os.path.join(dirpath, fname))
    return image_paths

def calculate_metrics(original, compressed):
    if original.ndim == 3 and original.shape[2] == 3:
        gray_orig = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
        gray_comp = cv2.cvtColor(compressed, cv2.COLOR_BGR2GRAY)
    else:
        gray_orig = original
        gray_comp = compressed

    psnr_val = psnr(original, compressed, data_range=255)
    ssim_val = ssim(gray_orig, gray_comp, data_range=255)
    mse_val = np.mean((original.astype(np.float32) - compressed.astype(np.float32)) ** 2)

    return psnr_val, ssim_val, mse_val

def main(original_root, compressed_root):
    original_images = get_all_image_paths(original_root)
    results = []

    for orig_path in tqdm(original_images, desc=f"Comparing images in {compressed_root}"):
        rel_path = os.path.relpath(orig_path, original_root)
        comp_path = os.path.join(compressed_root, rel_path)

        if not os.path.exists(comp_path):
            print(f"Missing compressed image: {comp_path}")
            continue

        orig_img = cv2.imread(orig_path)
        comp_img = cv2.imread(comp_path)

        if orig_img is None or comp_img is None:
            print(f"Could not read: {orig_path} or {comp_path}")
            continue

        if orig_img.shape != comp_img.shape:
            print(f"Shape mismatch: {orig_path}")
            continue

        psnr_val, ssim_val, mse_val = calculate_metrics(orig_img, comp_img)

        results.append({
            'file': rel_path,
            'PSNR': psnr_val,
            'SSIM': ssim_val,
            'MSE': mse_val
        })

    # Output summary (optional)
    avg_psnr = np.mean([r['PSNR'] for r in results]) if results else 0
    avg_ssim = np.mean([r['SSIM'] for r in results]) if results else 0
    avg_mse = np.mean([r['MSE'] for r in results]) if results else 0

    #print(f"\nSummary for {compressed_root}")
    print(f"Avg PSNR: {avg_psnr:.2f}, Avg SSIM: {avg_ssim:.4f}, Avg MSE: {avg_mse:.2f}\n")

    out = np.array([avg_psnr, avg_ssim, avg_mse])
    return out, results

if __name__ == '__main__':
    original_root = 'path_to_original_root'
    SNR_LIST = [8.0]  # You can add more
    RATIO_LIST = [1/4]  # Add more ratios if needed

    base_compressed_root = 'path_to_reconstructed_image'

    out_all = []
    for snr in SNR_LIST:
        for ratio in RATIO_LIST:
            prefix = f"Sentry_Data_snr{snr}_ratio{int(1/ratio)}"
            compressed_root = os.path.join(base_compressed_root, prefix)

            print(f"Processing SNR={snr}, Ratio={1/ratio}")
            out, results = main(original_root=original_root, compressed_root=compressed_root)
            out1 = np.append([snr, 1/ratio], out)
            out_all.append(out1)


